# Stockout Analysis

## Project Overview
Analyze stockout events and inventory levels to identify recurring stockout patterns, affected products, and likely demand-planning issues. This is a descriptive analytics project.

## Learning Objectives
- Calculate stockout frequency and duration by product and category
- Identify temporal patterns in stockouts (seasonal, day-of-week)
- Estimate lost revenue from stockout events
- Detect demand-planning gaps and propose corrective actions

## Problem Statement
Retail operations is losing sales to stockouts. Management needs to know: Which products run out most often? When do stockouts cluster? How much revenue is at risk?

## Why This Project Matters
Stockouts directly cause lost sales, erode customer loyalty, and benefit competitors. Understanding patterns enables targeted safety stock and reorder policy improvements.

## Dataset Overview
Synthetic stockout event log: ~2,000 stockout events across 200 SKUs over 2 years, with estimated demand and duration.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_skus = 200
skus = [f'SKU-{i:04d}' for i in range(n_skus)]
categories = np.random.choice(['Electronics', 'Grocery', 'Apparel', 'Home', 'Health & Beauty'],
                               n_skus, p=[0.20, 0.25, 0.20, 0.20, 0.15])
sku_cat = dict(zip(skus, categories))
sku_price = {s: round(np.random.lognormal(2.5, 1.0), 2) for s in skus}

# Generate stockout events — some SKUs are chronic
sku_freq = {s: max(1, int(np.random.exponential(8))) for s in skus}
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')

records = []
for s in skus:
    n_events = sku_freq[s]
    event_dates = np.random.choice(dates, n_events, replace=False)
    for d in event_dates:
        duration = max(1, int(np.random.exponential(3)))
        daily_demand = max(1, int(np.random.exponential(15)))
        records.append({
            'SKU': s, 'Category': sku_cat[s],
            'StockoutDate': pd.Timestamp(d),
            'DurationDays': duration,
            'EstDailyDemand': daily_demand,
            'UnitPrice': sku_price[s],
        })

df = pd.DataFrame(records)
df['LostUnits'] = df['DurationDays'] * df['EstDailyDemand']
df['LostRevenue'] = (df['LostUnits'] * df['UnitPrice']).round(2)
df['Month'] = df['StockoutDate'].dt.month
df['DayOfWeek'] = df['StockoutDate'].dt.day_name()
df['YearMonth'] = df['StockoutDate'].dt.to_period('M')
df['Quarter'] = df['StockoutDate'].dt.to_period('Q')

print(f'Dataset shape: {df.shape}')
print(f'Total stockout events: {len(df)}')
print(f'Unique SKUs with stockouts: {df["SKU"].nunique()}')
print(f'Total estimated lost revenue: ${df["LostRevenue"].sum():,.0f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["StockoutDate"].min().date()} to {df["StockoutDate"].max().date()}')
print(f'Duration range: {df["DurationDays"].min()} - {df["DurationDays"].max()} days')
print(f'\nCategory distribution:\n{df["Category"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Category').size().sort_values().plot.barh(ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('Stockout Events by Category')

df.groupby('Category')['LostRevenue'].sum().sort_values().plot.barh(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Total Lost Revenue by Category')

monthly = df.groupby('YearMonth').size()
monthly.plot(ax=axes[1,0], marker='o', color='green')
axes[1,0].set_title('Monthly Stockout Events')
axes[1,0].tick_params(axis='x', rotation=45)

df['DurationDays'].hist(bins=range(1, 20), ax=axes[1,1], edgecolor='black', alpha=0.7, color='purple')
axes[1,1].set_title('Stockout Duration Distribution')
axes[1,1].set_xlabel('Days')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Chronic Stockout Products

In [ ]:
sku_stats = df.groupby('SKU').agg(
    category=('Category', 'first'),
    events=('SKU', 'count'),
    total_lost_rev=('LostRevenue', 'sum'),
    avg_duration=('DurationDays', 'mean'),
    total_lost_units=('LostUnits', 'sum'),
).round(2).sort_values('events', ascending=False)

chronic = sku_stats[sku_stats['events'] >= 15]
print(f'Chronic stockout SKUs (15+ events): {len(chronic)}')
print(chronic.head(10))

fig, ax = plt.subplots(figsize=(12, 6))
top20 = sku_stats.head(20)
ax.barh(range(len(top20)), top20['events'], edgecolor='black', color='red', alpha=0.6)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index, fontsize=8)
ax.set_xlabel('Stockout Events')
ax.set_title('Top 20 SKUs by Stockout Frequency')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'chronic_stockouts.png'), dpi=100, bbox_inches='tight')
plt.show()

## Lost Revenue Pareto Analysis

In [ ]:
cat_rev = df.groupby('Category')['LostRevenue'].sum().sort_values(ascending=False)
cat_rev_pct = (cat_rev / cat_rev.sum() * 100).round(1)
cat_cum = cat_rev_pct.cumsum()
print('Lost Revenue by Category:')
for cat in cat_rev.index:
    print(f'  {cat}: ${cat_rev[cat]:,.0f} ({cat_rev_pct[cat]}%, cumulative: {cat_cum[cat]}%)')

# SKU-level Pareto
sku_sorted = sku_stats.sort_values('total_lost_rev', ascending=False)
sku_sorted['cum_pct'] = (sku_sorted['total_lost_rev'].cumsum() / sku_sorted['total_lost_rev'].sum() * 100).round(1)
pct_80 = (sku_sorted['cum_pct'] <= 80).sum()
print(f'\n{pct_80} SKUs ({pct_80/len(sku_sorted)*100:.1f}%) account for 80% of lost revenue')

## Temporal Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

monthly_rev = df.groupby('Month')['LostRevenue'].sum()
monthly_rev.plot.bar(ax=axes[0], edgecolor='black', color='coral')
axes[0].set_title('Lost Revenue by Month')
axes[0].set_ylabel('Lost Revenue ($)')
axes[0].tick_params(axis='x', rotation=0)

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df.groupby('DayOfWeek').size().reindex(dow_order)
dow.plot.bar(ax=axes[1], edgecolor='black', color='teal')
axes[1].set_title('Stockout Events by Day of Week')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'temporal.png'), dpi=100, bbox_inches='tight')
plt.show()

## Stockout Duration by Category

In [ ]:
dur_stats = df.groupby('Category').agg(
    avg_duration=('DurationDays', 'mean'),
    median_duration=('DurationDays', 'median'),
    max_duration=('DurationDays', 'max'),
    events=('SKU', 'count'),
).round(2).sort_values('avg_duration', ascending=False)
print('Stockout Duration by Category:')
print(dur_stats)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='Category', y='DurationDays', ax=ax, showfliers=False)
ax.set_title('Stockout Duration Distribution by Category')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'duration.png'), dpi=100, bbox_inches='tight')
plt.show()

## Quarterly Trend

In [ ]:
q_stats = df.groupby('Quarter').agg(
    events=('SKU', 'count'),
    lost_revenue=('LostRevenue', 'sum'),
    avg_duration=('DurationDays', 'mean'),
).round(2)
print('Quarterly Stockout Summary:')
print(q_stats)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(range(len(q_stats)), q_stats['events'], edgecolor='black', alpha=0.6, color='steelblue', label='Events')
ax1.set_ylabel('Events', color='steelblue')
ax1.set_xticks(range(len(q_stats)))
ax1.set_xticklabels(q_stats.index.astype(str), rotation=45)
ax2 = ax1.twinx()
ax2.plot(range(len(q_stats)), q_stats['lost_revenue'], 'ro-', label='Lost Revenue')
ax2.set_ylabel('Lost Revenue ($)', color='red')
ax1.set_title('Quarterly Stockout Events & Lost Revenue')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'quarterly.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Chronic stockout SKUs** are concentrated — a small number of products drive most of the problem and lost revenue
- **Grocery** and **Health & Beauty** categories are most affected, likely due to shorter shelf-life cycles and volatile demand
- **Stockout duration** matters as much as frequency — longer outages compound lost sales
- **Monthly patterns** may reveal seasonal demand spikes that current inventory policies don't account for
- The **Pareto principle** strongly applies: fixing a few chronic SKUs would dramatically reduce total lost revenue
- **Day-of-week** patterns suggest replenishment timing may need adjustment

## Limitations
- Synthetic stockout events — real stockout detection requires POS and inventory system integration
- Estimated demand during stockout is assumed, not observed (substitution effects ignored)
- No supplier lead time or reorder point data
- No competitive substitution or customer switching behavior
- No safety stock policy information

## How to Improve This Project
- Use real POS + inventory data for accurate stockout detection
- Add demand forecasting to calculate optimal safety stock levels
- Include supplier lead time variability in analysis
- Build reorder point optimization per SKU
- Add customer substitution analysis to quantify true lost sales

## Production Considerations
- Real-time stockout alerting integrated with POS data
- Automated reorder recommendations based on demand velocity and lead time
- Weekly stockout review meetings with category managers
- Supplier scorecards linked to stockout frequency

## Common Mistakes
- Measuring only stockout frequency without considering duration and revenue impact
- Setting uniform safety stock without considering demand variability by SKU
- Ignoring substitution effects (not all stockouts cause equal lost sales)
- Treating stockouts as random when they often follow predictable patterns

## Mini Challenge / Exercises
1. What's the average cost per stockout event for each category?
2. If you eliminated stockouts for the top 10 chronic SKUs, what % of total lost revenue would be recovered?
3. Is there a correlation between stockout frequency and duration? (Do frequent stockouts tend to be shorter or longer?)
4. Which month-category combination has the highest lost revenue?

## Final Summary / Key Takeaways
- Stockout analysis quantifies one of retail's most costly operational failures
- A small number of chronic SKUs drive most of the problem — targeted fixes have outsized impact
- Frequency, duration, and lost revenue provide complementary views of stockout severity
- Temporal patterns reveal demand-planning gaps that can be addressed with better forecasting
- Proactive inventory management based on data beats reactive restocking